# Project 2: Resume Job Fit Analyzer

**30 min project**

Build a multi-agent hiring pipeline that analyzes a resume against a job description.

**Concepts:** handoffs, specialist agents, parallel orchestration, dynamic instructions, chaining

In [ ]:
!pip install -q openai-agents python-dotenv

In [ ]:
from dotenv import load_dotenv

load_dotenv("../../.env")

## 1. Sample Data

In [ ]:
job_description = """
Senior Backend Engineer
We need a backend engineer with 5+ years of experience building Python APIs, distributed systems, PostgreSQL, Redis, cloud infrastructure, observability, and production incident response. Experience mentoring engineers and leading technical projects is important.
"""

resume_a = """
Asha Rao
Backend Engineer with 6 years of experience building Python and FastAPI services at scale.
Led migration from monolith to microservices, owned PostgreSQL schema design, Redis caching, CI/CD, Docker, Kubernetes, and Datadog observability.
Mentored 3 junior engineers and led incident response for high-traffic systems.
Skills: Python, FastAPI, PostgreSQL, Redis, AWS, Docker, Kubernetes, Datadog, Terraform.
"""

resume_b = """
Ben Lee
Full-stack developer with 4 years of experience building React dashboards and Node.js APIs.
Worked on product analytics, frontend performance, and internal tools. Some exposure to PostgreSQL and AWS Lambda.
Collaborated with designers and product managers. Interested in backend engineering.
Skills: JavaScript, TypeScript, React, Node.js, PostgreSQL, AWS Lambda, CSS.
"""

candidate_input = f"JOB DESCRIPTION:\n{job_description}\n\nRESUME:\n{resume_a}"

## 2. Tools + Output Models

In [ ]:
import asyncio
import re
from dataclasses import dataclass
from pydantic import BaseModel, Field
from agents import Agent, Runner, RunContextWrapper, function_tool


@function_tool
def extract_skills(text: str) -> list[str]:
    """Extract known engineering skills from text."""
    known = [
        "python", "fastapi", "django", "postgresql", "redis", "aws", "docker",
        "kubernetes", "terraform", "datadog", "react", "node.js", "typescript",
        "javascript", "ci/cd", "observability", "microservices",
    ]
    text_lower = text.lower()
    return [skill for skill in known if skill in text_lower]


@function_tool
def keyword_overlap(resume: str, job_description: str) -> float:
    """Calculate rough keyword overlap between resume and job description."""
    resume_words = set(re.findall(r"[a-zA-Z][a-zA-Z0-9+/.-]+", resume.lower()))
    jd_words = set(re.findall(r"[a-zA-Z][a-zA-Z0-9+/.-]+", job_description.lower()))
    useful_jd_words = {w for w in jd_words if len(w) > 3}
    return round(len(resume_words & useful_jd_words) / max(len(useful_jd_words), 1), 2)


@function_tool
def parse_experience_years(text: str) -> int:
    """Extract the highest mentioned years of experience."""
    matches = re.findall(r"(\d+)\+?\s+years?", text.lower())
    return max([int(m) for m in matches], default=0)


@function_tool
def check_leadership_signals(text: str) -> list[str]:
    """Find leadership signals in resume text."""
    signals = ["led", "owned", "mentored", "managed", "designed", "architected", "incident response"]
    text_lower = text.lower()
    return [s for s in signals if s in text_lower]


class SpecialistReport(BaseModel):
    area: str
    score: int = Field(description="Score from 1 to 10")
    strengths: list[str]
    gaps: list[str]


class FitReport(BaseModel):
    candidate_name: str
    overall_score: int = Field(description="Score from 1 to 10")
    skill_match: int = Field(description="Score from 1 to 10")
    experience_match: int = Field(description="Score from 1 to 10")
    culture_match: int = Field(description="Score from 1 to 10")
    strengths: list[str]
    gaps: list[str]
    recommendation: str


class InterviewQuestions(BaseModel):
    technical_questions: list[str]
    behavioral_questions: list[str]
    risk_areas_to_probe: list[str]

## 3. Specialist Agents

In [ ]:
skills_agent = Agent(
    name="Skills Analyzer",
    handoff_description="Evaluates technical skill match between resume and job description.",
    instructions="Focus only on technical skill match. Use tools. Return a SpecialistReport.",
    model="gpt-4o-mini",
    tools=[extract_skills, keyword_overlap],
    output_type=SpecialistReport,
)

experience_agent = Agent(
    name="Experience Analyzer",
    handoff_description="Evaluates seniority, relevant years of experience, and role progression.",
    instructions="Focus only on experience depth and seniority. Use tools. Return a SpecialistReport.",
    model="gpt-4o-mini",
    tools=[parse_experience_years, check_leadership_signals],
    output_type=SpecialistReport,
)

culture_agent = Agent(
    name="Culture Fit Analyzer",
    handoff_description="Evaluates collaboration, mentoring, ownership, and team fit signals.",
    instructions="Focus only on culture and collaboration signals. Use tools. Return a SpecialistReport.",
    model="gpt-4o-mini",
    tools=[check_leadership_signals],
    output_type=SpecialistReport,
)

## 4. Pattern A: Handoff Triage

In [ ]:
triage_agent = Agent(
    name="Resume Triage Agent",
    instructions=(
        "Route the user's resume evaluation question to exactly one specialist. "
        "Use Skills Analyzer for technical skill questions, Experience Analyzer for seniority questions, "
        "and Culture Fit Analyzer for collaboration or leadership questions. Do not answer directly."
    ),
    model="gpt-4o-mini",
    handoffs=[skills_agent, experience_agent, culture_agent],
)

result = Runner.run_sync(
    triage_agent,
    f"Is this candidate technically strong for the role?\n\n{candidate_input}",
)

print("Handled by:", result.last_agent.name)
print(result.final_output.model_dump_json(indent=2))

## 5. Pattern B: Parallel Orchestration

In [ ]:
@dataclass
class RoleContext:
    role_type: str


def synthesis_instructions(ctx: RunContextWrapper[RoleContext], agent: Agent) -> str:
    return (
        f"You synthesize specialist hiring reports for a {ctx.context.role_type}. "
        "Prioritize evidence from the resume and job description. Return a FitReport."
    )


synthesis_agent = Agent[RoleContext](
    name="Fit Report Synthesizer",
    instructions=synthesis_instructions,
    model="gpt-4o-mini",
    output_type=FitReport,
)


async def analyze_candidate(resume: str, job_description: str, role_type: str) -> FitReport:
    candidate_input = f"JOB DESCRIPTION:\n{job_description}\n\nRESUME:\n{resume}"

    skills_result, experience_result, culture_result = await asyncio.gather(
        Runner.run(skills_agent, candidate_input),
        Runner.run(experience_agent, candidate_input),
        Runner.run(culture_agent, candidate_input),
    )

    specialist_reports = f"""
Skills Report:
{skills_result.final_output.model_dump_json(indent=2)}

Experience Report:
{experience_result.final_output.model_dump_json(indent=2)}

Culture Report:
{culture_result.final_output.model_dump_json(indent=2)}
"""

    result = await Runner.run(
        synthesis_agent,
        f"Create the final fit report.\n\n{candidate_input}\n\n{specialist_reports}",
        context=RoleContext(role_type=role_type),
    )
    return result.final_output


fit_report = await analyze_candidate(resume_a, job_description, "senior backend engineer")
print(fit_report.model_dump_json(indent=2))

## 6. Agent Cloning for Role Variants

In [ ]:
manager_synthesis_agent = synthesis_agent.clone(
    name="Engineering Manager Fit Synthesizer",
    instructions=(
        "You synthesize specialist hiring reports for an engineering manager role. "
        "Prioritize people leadership, cross-functional communication, ownership, and team development. "
        "Return a FitReport."
    ),
)

manager_result = await Runner.run(
    manager_synthesis_agent,
    f"Re-evaluate this candidate for an engineering manager path:\n\n{fit_report.model_dump_json(indent=2)}",
)

print(manager_result.final_output.model_dump_json(indent=2))

## 7. Chain Into Interview Questions

In [ ]:
question_agent = Agent(
    name="Interview Question Generator",
    instructions=(
        "Generate targeted interview questions based on the candidate's gaps and risks. "
        "Questions should help validate whether the candidate can succeed in the role."
    ),
    model="gpt-4o-mini",
    output_type=InterviewQuestions,
)

question_result = await Runner.run(
    question_agent,
    f"Generate interview questions for this fit report:\n\n{fit_report.model_dump_json(indent=2)}",
)

questions = question_result.final_output
print("Technical questions:")
for q in questions.technical_questions:
    print("-", q)
print("\nBehavioral questions:")
for q in questions.behavioral_questions:
    print("-", q)
print("\nRisk areas:", questions.risk_areas_to_probe)

## 8. Compare Two Candidates

In [ ]:
report_a, report_b = await asyncio.gather(
    analyze_candidate(resume_a, job_description, "senior backend engineer"),
    analyze_candidate(resume_b, job_description, "senior backend engineer"),
)

reports = [report_a, report_b]
for report in reports:
    print(f"{report.candidate_name}: {report.overall_score}/10 - {report.recommendation}")
    print("  Strengths:", "; ".join(report.strengths[:2]))
    print("  Gaps:", "; ".join(report.gaps[:2]))
    print()

winner = max(reports, key=lambda r: r.overall_score)
print("Recommended candidate:", winner.candidate_name)